# ANN (Artificial Neural Network)

In [33]:
!pip install tensorflow


Defaulting to user installation because normal site-packages is not writeable


In [34]:
import sys
!{sys.executable} -m pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable


In [35]:
!pip install keras

Defaulting to user installation because normal site-packages is not writeable


In [1]:
import tensorflow as tf
from tensorflow import keras

print("TensorFlow Version:", tf.__version__)
print("Keras Version:", keras.__version__)

TensorFlow Version: 2.21.0
Keras Version: 3.15.0


In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

import warnings
warnings.filterwarnings("ignore")

In [2]:
dataset=pd.read_csv("credit_card_fraud_dataset.csv")


In [3]:
dataset.head()

,TransactionID,TransactionDate,Amount,MerchantID,TransactionType,Location,IsFraud
0,1,2024-04-03 14:15:35.462794,4189.27,688,refund,San Antonio,0
1,2,2024-03-19 13:20:35.462824,2659.71,109,refund,Dallas,0
2,3,2024-01-08 10:08:35.462834,784.00,394,purchase,New York,0
3,4,2024-04-13 23:50:35.462850,3514.40,944,purchase,Philadelphia,0
4,5,2024-07-12 18:51:35.462858,369.07,475,purchase,Phoenix,0


In [4]:
dataset.shape


(100000, 7)

In [5]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   TransactionID    100000 non-null  int64  
 1   TransactionDate  100000 non-null  object 
 2   Amount           100000 non-null  float64
 3   MerchantID       100000 non-null  int64  
 4   TransactionType  100000 non-null  object 
 5   Location         100000 non-null  object 
 6   IsFraud          100000 non-null  int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 5.3+ MB


In [7]:
dataset.isnull().sum()

TransactionID      0
TransactionDate    0
Amount             0
MerchantID         0
TransactionType    0
Location           0
IsFraud            0
dtype: int64

In [8]:
dataset["IsFraud"].value_counts()

IsFraud
0    99000
1     1000
Name: count, dtype: int64

In [10]:
dataset["IsFraud"].value_counts(normalize=True)*100

IsFraud
0    99.0
1     1.0
Name: proportion, dtype: float64

In [12]:
dataset["TransactionDate"] = pd.to_datetime(dataset["TransactionDate"])

In [13]:
dataset["Year"] = dataset["TransactionDate"].dt.year
dataset["Month"] = dataset["TransactionDate"].dt.month
dataset["Day"] = dataset["TransactionDate"].dt.day
dataset["Hour"] = dataset["TransactionDate"].dt.hour


In [14]:
from sklearn.preprocessing import LabelEncoder
le_type = LabelEncoder()
le_location = LabelEncoder()
dataset["TransactionType"]=le_type.fit_transform(dataset["TransactionType"])
dataset["Location"] = le_location.fit_transform(dataset["Location"])

In [15]:
dataset.drop(["TransactionID","TransactionDate"],axis=1,inplace=True)

In [16]:
dataset.head()

,Amount,MerchantID,TransactionType,Location,IsFraud,Year,Month,Day,Hour
0,4189.27,688,1,7,0,2024,4,3,14
1,2659.71,109,1,1,0,2024,3,19,13
2,784.00,394,0,4,0,2024,1,8,10
3,3514.40,944,0,5,0,2024,4,13,23
4,369.07,475,0,6,0,2024,7,12,18


In [18]:
X = dataset.drop("IsFraud", axis=1)
y = dataset["IsFraud"]

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
    

In [22]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train =scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [23]:
from imblearn.over_sampling import SMOTE
smote =SMOTE(random_state=42)
X_train,y_train=smote.fit_resample(X_train,y_train)
print(X_train.shape)
print(y_train.value_counts())

(158400, 8)
IsFraud
0    79200
1    79200
Name: count, dtype: int64


In [25]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential()

model.add(Dense(64, activation="relu", input_shape=(X_train.shape[1],)))
model.add(Dropout(0.3))

model.add(Dense(32, activation="relu"))
model.add(Dropout(0.2))

model.add(Dense(16, activation="relu"))

model.add(Dense(1, activation="sigmoid"))

In [27]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [28]:
history= model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    verbose=1
)

Epoch 1/20
3960/3960 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - accuracy: 0.6268 - loss: 0.6453 - val_accuracy: 0.1662 - val_loss: 0.8522
Epoch 2/20
3960/3960 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.6400 - loss: 0.6246 - val_accuracy: 0.3997 - val_loss: 0.7889
Epoch 3/20
3960/3960 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.6545 - loss: 0.6102 - val_accuracy: 0.5689 - val_loss: 0.7108
Epoch 4/20
3960/3960 ━━━━━━━━━━━━━━━━━━━━ 17s 4ms/step - accuracy: 0.6649 - loss: 0.5997 - val_accuracy: 0.5368 - val_loss: 0.7266
Epoch 5/20
3960/3960 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - accuracy: 0.6723 - loss: 0.5904 - val_accuracy: 0.5911 - val_loss: 0.6976
Epoch 6/20
3960/3960 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.6777 - loss: 0.5832 - val_accuracy: 0.6075 - val_loss: 0.6801
Epoch 7/20
3960/3960 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.6831 - loss: 0.5775 - val_accuracy: 0.6367 - val_loss: 0.6674
Epoch 8/20
3960/3960 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - accuracy: 0.6894 - loss: 0

In [29]:
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)

625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [30]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.76      0.86     19800
           1       0.01      0.28      0.02       200

    accuracy                           0.75     20000
   macro avg       0.50      0.52      0.44     20000
weighted avg       0.98      0.75      0.85     20000



In [31]:
import pickle
model.save("fraud_model.keras")

pickle.dump(scaler, open("scaler.pkl", "wb"))

pickle.dump(
    {
        "TransactionType": le_type,
        "Location": le_location
    },
    open("label_encoders.pkl", "wb")
)

print("Model Saved Successfully")

Model Saved Successfully


In [ ]:
*-